# 03 — Inspect the matching block-0 LiteRT graph
Read the frozen model selected by `workflow_config.MODEL_PATH`, which defaults to the recovered
`block0_w8a8_baseline.tflite`. Save `block0_graph.json`, `block0_matrix_ops.csv`, source snapshots,
and a manifest into a unique `results/notebook_runs/<run-id>/` directory.
This step uses the LiteRT interpreter; it does not download the PyTorch model.
Use actual `op['index']` values. Operation/tensor indices may change after reconversion.

Start from a fresh kernel and run top to bottom. The final cell records output hashes; incomplete runs remain `started`.


In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the repository root or notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/python/workflow_config.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Start Jupyter in the extracted tinystories-rv64imc folder.')
sys.path.insert(0, str(ROOT / 'src/python'))
from workflow_config import (MODEL_ID, MODEL_REVISION, SEQUENCE_LENGTH,
                             MODEL_DIR, EXPORT_DIR, RESULT_DIR, C_DIR, MODEL_PATH,
                             PROMPT, GENERATED_LENGTH)
from notebook_runs import start_run, finish_run
RUN_DIR = start_run(ROOT, 'notebooks/03_inspect_block0_graph.ipynb', {'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'sequence_length': SEQUENCE_LENGTH}, inputs=[MODEL_PATH])
RESULT_DIR = RUN_DIR


In [ ]:
import csv
import hashlib
import json
from collections import Counter
from litert_helpers import reference_interpreter, tensor_record

interpreter = reference_interpreter(MODEL_PATH)
tensors = {int(t['index']): tensor_record(t) for t in interpreter.get_tensor_details()}
# Private debugging API; validate again after changing interpreter versions.
ops = interpreter._get_ops_details()
print('Input tensors:', [tensors[int(t['index'])] for t in interpreter.get_input_details()])
print('Output tensors:', [tensors[int(t['index'])] for t in interpreter.get_output_details()])
print('Tensor dtypes:', dict(Counter(t['dtype'] for t in tensors.values())))
print('Operator counts:', dict(Counter(op['op_name'] for op in ops)))

## Matrix operations and quantization metadata

In [ ]:
matrix_rows = []
for op in ops:
    if op['op_name'] not in {'FULLY_CONNECTED', 'BATCH_MATMUL'}:
        continue
    inputs = [tensors[int(i)] for i in op['inputs'] if int(i) >= 0]
    outputs = [tensors[int(i)] for i in op['outputs'] if int(i) >= 0]
    w8a8 = (len(inputs) >= 2 and inputs[0]['dtype'] == inputs[1]['dtype'] == 'int8'
            and bool(outputs) and all(t['dtype'] == 'int8' for t in outputs))
    print(f"\nOp {int(op['index'])}: {op['op_name']}; INT8 inputs/output: {w8a8}")
    for label, records in [('input', inputs), ('output', outputs)]:
        for t in records:
            print(f"  {label} T{t['index']}: {t['shape']} {t['dtype']}; "
                  f"scales={len(t['scales'])}; zero_points={t['zero_points']}")
    matrix_rows.append({'op_index': int(op['index']), 'op_name': op['op_name'],
                        'inputs': json.dumps([int(i) for i in op['inputs']]),
                        'outputs': json.dumps([int(i) for i in op['outputs']]),
                        'int8_inputs_and_output': w8a8})

## Save full metadata for the block-0 analysis

In [ ]:
RESULT_DIR.mkdir(parents=True, exist_ok=True)
graph = {
    'model_sha256': hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest(),
    'tensors': list(tensors.values()),
    'ops': [{'index': int(op['index']), 'name': op['op_name'],
             'inputs': [int(i) for i in op['inputs']],
             'outputs': [int(i) for i in op['outputs']]} for op in ops],
}
(RESULT_DIR / 'block0_graph.json').write_text(json.dumps(graph, indent=2))
with (RESULT_DIR / 'block0_matrix_ops.csv').open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['op_index', 'op_name', 'inputs', 'outputs',
                                         'int8_inputs_and_output'])
    writer.writeheader()
    writer.writerows(matrix_rows)
print('Saved graph JSON and matrix-op CSV')

## Interpret the next step
Inspect normalization, attention matmuls, softmax, residual additions, and MLP operations
using the saved graph. INT8 matrix operators alone do not establish integer full-block execution.
The baseline q-projection extraction expects Op15, T45/T27 → T46; notebook 04 checks that layout.

In [ ]:
finish_run(ROOT, RUN_DIR, [RESULT_DIR / 'block0_graph.json', RESULT_DIR / 'block0_matrix_ops.csv'])
print('Saved run record:', RUN_DIR / 'run.json')
